# Chapter 11: Novelty and Outlier Detection

## Ringkasan Bab & Penjelasan Teoretis
Deteksi anomali terbagi menjadi dua skenario utama:
1. **Outlier Detection (Unsupervised)**: Data latih sudah tercemar oleh pengamatan ekstrem (*outliers*), dan algoritma berusaha memisahkan *inliers* ($1$) dari *outliers* ($-1$).
2. **Novelty Detection (Semi-Supervised)**: Data latih bersih dan hanya berisi pengamatan normal (*inliers*); model mempelajari batas distribusi normal untuk mendeteksi apakah sampel baru di masa depan merupakan anomali (*novelty*).

### Algoritma Utama di `scikit-learn`
* **Isolation Forest (`IsolationForest`)**: Mengisolasi pengamatan dengan memilih fitur dan nilai pemisahan secara acak di dalam struktur pohon. Karena *outlier* memiliki nilai yang ekstrem dan jarang, mereka membutuhkan jumlah pemisahan (panjang jalur dari akar ke daun) yang jauh lebih sedikit dibandingkan titik normal.
* **One-Class SVM (`OneClassSVM`)**: Mempelajari batas wilayah yang membungkus data normal di ruang fitur menggunakan fungsi kernel (seperti RBF). Sangat cocok untuk *novelty detection*.
* **Local Outlier Factor (`LocalOutlierFactor` / LOF)**: Mengukur deviasi kepadatan lokal suatu titik data relatif terhadap $k$ tetangga terdekatnya. Titik dengan kepadatan lokal yang jauh lebih rendah daripada tetangganya diidentifikasi sebagai *outlier*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, f1_score

# Membuat Dataset Sintetis dengan Inliers (1) dan Outliers (-1)
np.random.seed(2024)
X_inliers = 0.3 * np.random.randn(200, 2)
X_inliers = np.r_[X_inliers + 2, X_inliers - 2]  # 400 titik normal dalam 2 klaster
X_outliers = np.random.uniform(low=-4, high=4, size=(40, 2))  # 40 titik outlier tersebar

X = np.r_[X_inliers, X_outliers]
y_true = np.array([1] * len(X_inliers) + [-1] * len(X_outliers))  # 1 = Inlier, -1 = Outlier

# 1. Isolation Forest
iso_forest = IsolationForest(contamination=0.09, random_state=2024)
y_pred_iso = iso_forest.fit_predict(X)

# 2. Local Outlier Factor (LOF)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.09)
y_pred_lof = lof.fit_predict(X)

# 3. One-Class SVM (Dilatih pada data normal untuk Novelty Detection, lalu memprediksi seluruh X)
oc_svm = OneClassSVM(kernel='rbf', gamma=0.1, nu=0.05)
oc_svm.fit(X_inliers)
y_pred_svm = oc_svm.predict(X)

# Evaluasi Perbandingan Ketiga Metode Deteksi Anomali
comparison_df = pd.DataFrame({
    'Method': ['Isolation Forest', 'Local Outlier Factor (LOF)', 'One-Class SVM'],
    'Outliers Detected': [(y_pred_iso == -1).sum(), (y_pred_lof == -1).sum(), (y_pred_svm == -1).sum()],
    'F1-Score (Outlier Class = -1)': [
        f1_score(y_true, y_pred_iso, pos_label=-1),
        f1_score(y_true, y_pred_lof, pos_label=-1),
        f1_score(y_true, y_pred_svm, pos_label=-1)
    ]
})
display(comparison_df)

# Visualisasi Hasil Deteksi Inliers vs Outliers
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
preds = [y_pred_iso, y_pred_lof, y_pred_svm]
titles = ['Isolation Forest', 'Local Outlier Factor (LOF)', 'One-Class SVM']

for ax, pred, title in zip(axes, preds, titles):
    ax.scatter(X[pred == 1, 0], X[pred == 1, 1], c='royalblue', edgecolor='k', s=30, label='Inliers (1)')
    ax.scatter(X[pred == -1, 0], X[pred == -1, 1], c='crimson', marker='x', s=60, label='Outliers (-1)')
    ax.set_title(title)
    ax.legend(loc='upper left')
plt.tight_layout()
plt.show()